# AssureX Claim Engine — Exploratory Data Analysis (EDA)

**Project:** AssureX Claim Engine  
**Theme:** AI-Powered Document Ops  
**Category:** NextWave AI and ML

This notebook performs a full exploratory data analysis on the AssureX warranty claim dataset.  
The dataset contains **1,500 synthetic warranty claim records** with three balanced classes:
- `valid_claim` — 500 records
- `invalid_claim` — 500 records  
- `manual_review` — 500 records

**SRS Reference:** Section 1.2, Steps 3–4 (Dataset Creation, Data Pre-Processing)


## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, '..')   # Add project root to path

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")


Libraries loaded successfully.
  pandas  : 2.2.0
  numpy   : 1.26.0


## 2. Load Dataset

In [ ]:
# Load the full 1500-record dataset
df = pd.read_csv('../data/raw/assurex_claims_full.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


In [ ]:
# Preview first 5 rows
df.head()


In [ ]:
# Dataset info
df.info()


## 3. Class Distribution

In [ ]:
label_counts = df['label'].value_counts()
print("Class distribution:")
print(label_counts)
print(f"\nBalance check: {label_counts.std():.1f} std (0 = perfectly balanced)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#16A34A', '#DC2626', '#F59E0B']
label_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Claim Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Invalid Claim', 'Manual Review', 'Valid Claim'], rotation=15)
for bar, count in zip(axes[0].patches, label_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, shadow=True)
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → reports/eda_class_distribution.png")


## 4. Feature Analysis

### 4.1 Numerical Features

In [ ]:
num_cols = ['product_age_months', 'remaining_warranty_days', 'days_since_fault',
            'claim_submission_delay_days', 'repair_count', 'missing_doc_count', 'purchase_price']

print("Numerical Feature Statistics:")
df[num_cols].describe().round(2)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color in zip(['valid_claim', 'invalid_claim', 'manual_review'],
                            ['#16A34A', '#DC2626', '#F59E0B']):
        subset = df[df['label'] == label][col]
        axes[i].hist(subset, bins=20, alpha=0.6, color=color, label=label.replace('_', ' ').title())
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=7)

# Hide the extra subplot
axes[-1].set_visible(False)
plt.suptitle('Numerical Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → reports/eda_numerical_distributions.png")


### 4.2 Categorical Features

In [ ]:
cat_cols = ['damage_type', 'warranty_type', 'product_category']

for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=ax, color='#2563EB', edgecolor='black', alpha=0.8)
    ax.set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → reports/eda_categorical_distributions.png")


### 4.3 Binary / Flag Features

In [ ]:
bin_cols = ['has_purchase_receipt', 'has_warranty_card', 'has_product_image',
            'has_fault_evidence', 'has_repair_report', 'serial_number_match',
            'had_unauthorized_repair', 'is_duplicate_flag']

print("Binary Feature — Yes (1) rates per class:")
for col in bin_cols:
    rates = df.groupby('label')[col].mean().round(3) * 100
    print(f"  {col:<28}: Valid={rates.get('valid_claim',0):.1f}%  "
          f"Invalid={rates.get('invalid_claim',0):.1f}%  "
          f"Manual={rates.get('manual_review',0):.1f}%")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(bin_cols):
    rates = df.groupby('label')[col].mean() * 100
    rates = rates.reindex(['valid_claim', 'invalid_claim', 'manual_review'])
    bars = axes[i].bar(
        ['Valid', 'Invalid', 'Manual'],
        rates.values,
        color=['#16A34A', '#DC2626', '#F59E0B'],
        edgecolor='black'
    )
    for bar, val in zip(bars, rates.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.0f}%', ha='center', fontsize=8)
    axes[i].set_title(col.replace('_', ' ').replace('has ', '').title(), fontsize=9)
    axes[i].set_ylim(0, 115)
    axes[i].set_ylabel('% Yes')

plt.suptitle('Binary Feature Rates by Class (%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_binary_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → reports/eda_binary_features.png")


## 5. Correlation Analysis

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()
le = LabelEncoder()
df_enc['label_enc'] = le.fit_transform(df_enc['label'])

all_num = num_cols + bin_cols + ['label_enc']
corr = df_enc[all_num].corr()

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(all_num)))
ax.set_yticks(range(len(all_num)))
ax.set_xticklabels([c.replace('_', '\n') for c in all_num], fontsize=8, rotation=45, ha='right')
ax.set_yticklabels([c.replace('_', ' ') for c in all_num], fontsize=8)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved → reports/eda_correlation_heatmap.png")


## 6. Missing Values Check

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found. ✓")


## 7. Dataset Split Verification

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/validation.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

print("Split sizes:")
print(f"  Train      : {len(train_df):>4} records  ({len(train_df)/1500*100:.0f}%)")
print(f"  Validation : {len(val_df):>4} records  ({len(val_df)/1500*100:.0f}%)")
print(f"  Test       : {len(test_df):>4} records  ({len(test_df)/1500*100:.0f}%)")
print(f"  Total      : {len(train_df)+len(val_df)+len(test_df):>4} records")

print("\nClass balance per split:")
for name, sdf in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    counts = sdf['label'].value_counts()
    print(f"  {name}: {dict(counts)}")


## 8. EDA Summary

In [ ]:
print("=" * 55)
print("AssureX Dataset — EDA Summary")
print("=" * 55)
print(f"Total records     : {len(df)}")
print(f"Features          : {len(df.columns) - 1}")
print(f"Class balance     : Perfectly balanced (500 each)")
print(f"Missing values    : None")
print(f"Numerical features: {len(num_cols)}")
print(f"Categorical feat. : {len(cat_cols)}")
print(f"Binary features   : {len(bin_cols)}")
print()
print("Key observations:")
print("  • Valid claims:   high receipt rate, no unauthorized repairs, serial match")
print("  • Invalid claims: expired warranty, missing docs, excluded damage types")
print("  • Manual review:  borderline cases, contradictory data, multiple repairs")
print()
print("Dataset is ready for preprocessing and model training.")
print("=" * 55)
